In [ ]:
"../cvpr_results/coralcam_results_label_tolerance_0.csv"
import pandas as pd
df = pd.read_csv("../results/cvpr_results/coralcam_results_label_tolerance_0.csv")

In [ ]:
df.head()

In [ ]:
# --- Setup ---
import os, re, glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Directory and filename pattern
DATA_DIR = "../results/cvpr2"
PATTERN = "coralcam_results_label_tolerance_*.csv"  # adjust if your pattern differs

# --- Load & combine all tolerance files ---
files = sorted(glob.glob(os.path.join(DATA_DIR, PATTERN)))
if not files:
    raise FileNotFoundError(f"No files matched {os.path.join(DATA_DIR, PATTERN)}")

def parse_tolerance_from_path(path):
    """
    Extracts the numeric tolerance from filenames like:
    coralcam_results_label_tolerance_0.csv, coralcam_results_label_tolerance_0.25.csv
    """
    m = re.search(r"tolerance_([0-9]*\.?[0-9]+)", os.path.basename(path))
    if not m:
        raise ValueError(f"Could not parse tolerance from filename: {path}")
    return float(m.group(1))

dfs = []
for fp in files:
    tol = parse_tolerance_from_path(fp)
    tmp = pd.read_csv(fp)
    tmp["tolerance"] = tol
    dfs.append(tmp)

df = pd.concat(dfs, ignore_index=True)

# --- Infer metrics and config columns ---
# Numeric columns are candidates for metrics; we'll exclude the tolerance itself.
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
metric_cols = [c for c in numeric_cols if c != "tolerance"]

# --- Basic sanity checks ---
# Each config should have multiple tolerance points to make a line
counts = df.groupby("id")["tolerance"].nunique().sort_values(ascending=False)
print("\nTolerance counts per configuration (unique tolerances):")
display(counts)

# --- Plotting: one figure per configuration, each metric in its own subplot ---
# You can limit to top K configurations if there are many
MAX_CONFIGS = None  # e.g., set to 10 to limit
unique_cfgs = df["id"].unique().tolist()
if MAX_CONFIGS is not None:
    unique_cfgs = unique_cfgs[:MAX_CONFIGS]

# Ensure tolerance is sorted on the x-axis
def plot_config(config_name, metrics):
    sub = df[df["id"] == config_name].copy()
    sub = sub.sort_values("tolerance")

    # Handle duplicates (same tolerance repeated): average them
    sub_agg = sub.groupby("tolerance", as_index=False)[metrics].mean()

    n_metrics = len(metrics)
    # Arrange subplots in a neat grid
    ncols = min(3, n_metrics)
    nrows = int(np.ceil(n_metrics / ncols))
    fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(5*ncols, 3.2*nrows))
    if n_metrics == 1:
        axes = np.array([axes])
    axes = axes.flatten()

    for i, m in enumerate(metrics):
        ax = axes[i]
        ax.plot(sub_agg["tolerance"], sub_agg[m], marker="o")
        ax.set_xlabel("Tolerance")
        ax.set_ylabel(m)
        ax.set_title(m)
        ax.grid(True, linestyle="--", alpha=0.4)

    # Hide any unused axes
    for j in range(i+1, len(axes)):
        axes[j].axis("off")

    fig.suptitle(f"Effect of Tolerance on Metrics\n{config_name}", y=1.02, fontsize=12)
    fig.tight_layout()
    plt.show()

# Generate plots
for cfg in unique_cfgs:
    plot_config(cfg, metric_cols)